<a href="https://colab.research.google.com/github/garaimg/repositorios-git-pruebas/blob/main/test_LLMs_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✅ Validación de `rasbt/LLMs-from-scratch` en Google Colab

Este notebook comprueba si el repositorio [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)
(código del libro *Build a Large Language Model (From Scratch)* de Sebastian Raschka) funciona en Colab.

Reproduce lo que hace el propio CI del repositorio (GitHub Actions) y añade una prueba funcional real:

| Bloque | Qué prueba |
|---|---|
| **0. Entorno** | Versión de Python, PyTorch, GPU |
| **1. Instalación** | Clona el repo e instala dependencias + el paquete `llms_from_scratch` |
| **2. Check de entorno** | Script oficial `python_environment_check.py` contra `requirements.txt` |
| **3. Tests del libro** | `pytest` de los capítulos 4, 5 y 6 (los mismos que el CI) |
| **4. Notebooks** | Re-ejecuta notebooks de los capítulos 2 y 3 con `nbval` y compara salidas |
| **5. Bonus** | KV-cache, atención multi-cabeza eficiente, BPE desde cero, utilidades del cap. 7 |
| **6. Prueba funcional** | Carga los pesos reales de GPT-2 (124M) en el modelo del libro y genera texto |
| **7. Extendido (opcional)** | Tests del paquete `llms_from_scratch` y de Llama 3.2 / Qwen3 / Gemma 3 / OLMo 3 |
| **8. Resumen** | Tabla final con ✅ / ⚠️ / ❌ por prueba |

**Cómo usarlo:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)` (opcional pero recomendable)
y después `Entorno de ejecución → Ejecutar todo`. El resumen final aparece en la última celda.

⏱️ Tiempo aproximado: ~5–8 min con GPU T4 (más si activas el bloque extendido o usas CPU).

In [13]:
# ⚙️ Configuración
REPO_URL = "https://github.com/rasbt/LLMs-from-scratch.git"
REPO_DIR = "/content/LLMs-from-scratch"
REPO_REF = None          # p. ej. "main", un tag o un commit concreto; None = último commit de main
RUN_EXTENDED = True      # bloque 7: tests del paquete y de modelos bonus (más lento)
RUN_GPT2_DEMO = True     # bloque 6: descarga ~500 MB de pesos de GPT-2 124M

## 0. Entorno

In [2]:
import sys, platform, subprocess, os, time
import torch

print("Python  :", sys.version.split()[0])
print("Sistema :", platform.platform())
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    print("⚠️  Sin GPU: todo funcionará, pero los tests serán más lentos.")

Python  : 3.13.16
Sistema : Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch : 2.11.0+cu130
CUDA    : True
GPU     : Tesla T4


In [3]:
# Utilidad: ejecuta un comando, muestra su salida en directo y guarda el resultado para el resumen final
import re
RESULTS = []
ENV = {**os.environ, "PY_COLORS": "0", "NO_COLOR": "1"}
ANSI = re.compile(r"\x1b\[[0-9;]*m")
NET_ERR = re.compile(r"ProxyError|ConnectionError|Max retries exceeded|NameResolutionError|"
                     r"Temporary failure in name resolution|HTTPError: [45]\d\d|ReadTimeout")

def run_step(name, cmd, cwd=None, lax_cmd=None):
    # Ejecuta `cmd`. Si falla y hay `lax_cmd`, lo reintenta en modo tolerante (para notebooks).
    cwd = cwd or REPO_DIR
    print(f"\n{'='*90}\n▶ {name}\n$ {cmd}\n{'='*90}")
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, env=ENV,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(ANSI.sub("", line.rstrip()))
    proc.wait()
    ok = proc.returncode == 0
    # Detalle: la línea resumen de pytest ("3 passed in 2.1s") o la última línea
    summary = [l for l in lines if re.search(r"\d+ (passed|failed|error)", l)]
    detail = (summary[-1] if summary else (lines[-1] if lines else "")).strip("= ")
    if not ok:
        if proc.returncode in (-9, 137):
            detail = "Proceso terminado por el sistema (probablemente sin RAM)"
        elif any(NET_ERR.search(l) for l in lines):
            detail = "Error de red al descargar datos/pesos · " + detail
    status = "✅ OK" if ok else "❌ FALLO"
    if not ok and lax_cmd:
        print("\n↻ Reintentando en modo tolerante (ignora diferencias en las salidas)...")
        lax = subprocess.run(lax_cmd, shell=True, cwd=cwd, capture_output=True, text=True, env=ENV)
        print(lax.stdout[-2000:])
        if lax.returncode == 0:
            status = "⚠️ Ejecuta, salidas distintas"
            detail = "El código corre sin errores; difieren valores impresos (versión/hardware)"
    RESULTS.append({"Prueba": name, "Resultado": status,
                    "Tiempo (s)": round(time.time() - t0, 1), "Detalle": detail[:140]})
    return proc.returncode

## 1. Clonar el repositorio e instalar dependencias

In [4]:
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
if REPO_REF:
    !cd {REPO_DIR} && git fetch --depth 1 origin {REPO_REF} && git checkout -q FETCH_HEAD
%cd {REPO_DIR}
!git log -1 --format="Commit probado: %h · %ad · %s" --date=short

Cloning into '/content/LLMs-from-scratch'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (445/445), done.
remote: Compressing objects: 100% (362/362), done.
remote: Total 445 (delta 114), reused 299 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (445/445), 2.28 MiB | 19.42 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/content/LLMs-from-scratch
Commit probado: b5a5165 · 2026-11-06 · Fix training loss in hparam search summary (#1110)


In [5]:
# Colab ya trae torch, tensorflow, numpy, pandas, matplotlib, tqdm y requests.
# Solo faltan tiktoken (tokenizador de GPT-2), las herramientas de test y el propio paquete del libro.
!pip install -q tiktoken pytest nbval
!pip install -q --no-deps .          # instala `llms_from_scratch` desde la carpeta pkg/ del repo

import importlib; importlib.invalidate_caches()
import tiktoken, llms_from_scratch
print("tiktoken          :", tiktoken.__version__)
print("llms_from_scratch : instalado en", os.path.dirname(llms_from_scratch.__file__))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
tiktoken          : 0.14.0
llms_from_scratch : instalado en /usr/local/lib/python3.13/dist-packages/llms_from_scratch


## 2. Check de entorno oficial

Ejecuta `setup/02_installing-python-libraries/python_environment_check.py`, que compara las versiones instaladas
con `requirements.txt`. **JupyterLab se ignora a propósito**: Colab usa su propio frontend y
instalarlo puede romper el entorno de Colab.

In [6]:
print("▶ Check de entorno (python_environment_check.py)")
t0 = time.time()
out = subprocess.run([sys.executable, "setup/02_installing-python-libraries/python_environment_check.py"],
                     cwd=REPO_DIR, capture_output=True, text=True)
print(out.stdout, out.stderr)
fails = [l for l in out.stdout.splitlines() if "[FAIL]" in l and "jupyterlab" not in l.lower()]
RESULTS.append({"Prueba": "Check de entorno (requirements.txt)",
                "Resultado": "✅ OK" if not fails and out.returncode == 0 else "❌ FALLO",
                "Tiempo (s)": round(time.time() - t0, 1),
                "Detalle": "; ".join(fails)[:120] if fails else "jupyterlab ignorado (no aplica en Colab)"})

▶ Check de entorno (python_environment_check.py)
[OK] Your Python version is 3.13.16
[OK] torch 2.11.0+cu130
[FAIL] jupyterlab 0.0, please install a version matching >=4.0
[OK] tiktoken 0.14.0
[OK] matplotlib 3.10.0
[OK] tensorflow 2.21.0
[OK] tqdm 4.67.3
[OK] numpy 2.4.6
[OK] pandas 2.2.3
[OK] psutil 5.9.5
I0000 00:00:1793956956.731538    2779 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793956961.962403    2779 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.



## 3. Tests de los capítulos principales

Son los mismos que ejecuta el workflow `basic-tests-pip.yml` del repositorio:

- **Cap. 4** – arquitectura GPT y generación de texto
- **Cap. 5** – bucle de preentrenamiento (entrena un GPT pequeño con *The Verdict*) y disponibilidad de los pesos de GPT-2
- **Cap. 6** – *fine-tuning* para clasificación de spam (descarga el dataset SMS Spam Collection)

In [7]:
run_step("Cap. 4 · Modelo GPT",           "python -m pytest -q ch04/01_main-chapter-code/tests.py")
run_step("Cap. 5 · Preentrenamiento",     "python -m pytest -q ch05/01_main-chapter-code/tests.py")
run_step("Cap. 6 · Fine-tuning (spam)",   "python -m pytest -q ch06/01_main-chapter-code/tests.py")


▶ Cap. 4 · Modelo GPT
$ python -m pytest -q ch04/01_main-chapter-code/tests.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
.                                                                        [100%]
1 passed in 12.84s

▶ Cap. 5 · Preentrenamiento
$ python -m pytest -q ch05/01_main-chapter-code/tests.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
......                                                                   [100%]
6 passed in 38.89s

▶ Cap. 6 · Fine-tuning 

0

## 4. Validación de notebooks con `nbval`

`nbval` re-ejecuta cada celda y compara la salida con la que está guardada en el notebook.
Si falla el modo estricto, se reintenta con `--nbval-lax` para distinguir entre
**errores reales** (❌) y **pequeñas diferencias numéricas o de formato** (⚠️) debidas a otra versión de PyTorch o a otro hardware.

In [8]:
notebooks = {
    "Cap. 2 · Dataloader (notebook)":           "ch02/01_main-chapter-code/dataloader.ipynb",
    "Cap. 3 · Multi-head attention (notebook)": "ch03/01_main-chapter-code/multihead-attention.ipynb",
    "Cap. 2 · Intuición del dataloader (nb)":   "ch02/04_bonus_dataloader-intuition/dataloader-intuition.ipynb",
}
for name, nb in notebooks.items():
    run_step(name, f"python -m pytest -q -p no:cacheprovider --nbval {nb}",
             lax_cmd=f"python -m pytest -q -p no:cacheprovider --nbval-lax {nb}")


▶ Cap. 2 · Dataloader (notebook)
$ python -m pytest -q -p no:cacheprovider --nbval ch02/01_main-chapter-code/dataloader.ipynb
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
s...                                                                     [100%]
3 passed, 1 skipped in 3.57s

▶ Cap. 3 · Multi-head attention (notebook)
$ python -m pytest -q -p no:cacheprovider --nbval ch03/01_main-chapter-code/multihead-attention.ipynb
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
......

## 5. Material bonus

In [9]:
run_step("Cap. 3 · MHA eficiente (bonus)",
         "python -m pytest -q ch03/02_bonus_efficient-multihead-attention/tests/test_mha_implementations.py")
run_step("Cap. 4 · KV-cache (bonus)",      "python -m pytest -q ch04/03_kv-cache/tests.py")
run_step("Cap. 2 · BPE desde cero (bonus)", "python -m pytest -q ch02/05_bpe-from-scratch/tests.py")
run_step("Cap. 7 · Utilidades de dataset (bonus)",
         "pip install -q -r ch07/02_dataset-utilities/requirements-extra.txt && "
         "python -m pytest -q ch07/02_dataset-utilities/tests.py")


▶ Cap. 3 · MHA eficiente (bonus)
$ python -m pytest -q ch03/02_bonus_efficient-multihead-attention/tests/test_mha_implementations.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
....                                                                     [100%]
4 passed in 4.25s

▶ Cap. 4 · KV-cache (bonus)
$ python -m pytest -q ch04/03_kv-cache/tests.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
.........                                                                [100%]

0

## 6. Prueba funcional: GPT-2 real con el código del libro

Los tests unitarios usan modelos diminutos. Aquí se comprueba el flujo completo de los capítulos 4 y 5:
construir `GPTModel`, descargar los pesos oficiales de GPT-2 (124M), cargarlos con `load_weights_into_gpt`
y generar texto. Si el modelo está bien implementado, el texto tiene que ser inglés coherente (con pesos
aleatorios saldría basura).

In [10]:
if RUN_GPT2_DEMO:
    t0 = time.time()
    try:
        import tiktoken, torch
        from llms_from_scratch.ch04 import GPTModel
        from llms_from_scratch.ch05 import (download_and_load_gpt2, load_weights_into_gpt,
                                            generate, text_to_token_ids, token_ids_to_text)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        settings, params = download_and_load_gpt2(model_size="124M", models_dir="/content/gpt2")

        cfg = {"vocab_size": 50257, "context_length": 1024, "emb_dim": 768, "n_heads": 12,
               "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True}
        model = GPTModel(cfg)
        load_weights_into_gpt(model, params)
        model.to(device).eval()

        tokenizer = tiktoken.get_encoding("gpt2")
        torch.manual_seed(123)
        prompt = "Every effort moves you"
        ids = generate(model=model, idx=text_to_token_ids(prompt, tokenizer).to(device),
                       max_new_tokens=30, context_size=cfg["context_length"], top_k=25, temperature=1.4)
        text = token_ids_to_text(ids, tokenizer)
        print("\n📝 Texto generado:\n", text)

        # Comprobación objetiva: la pérdida de GPT-2 sobre un texto en inglés debe ser baja (< 6 nats; con pesos aleatorios sale ≈ 11)
        sample = ("The quick brown fox jumps over the lazy dog. Machine learning is a field of "
                  "artificial intelligence that uses statistical techniques to give computers the "
                  "ability to learn from data.")
        tok = text_to_token_ids(sample, tokenizer).to(device)
        with torch.no_grad():
            logits = model(tok[:, :-1])
            loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), tok[:, 1:].flatten()).item()
        print(f"\nPérdida (cross-entropy) en texto de control: {loss:.2f}  ·  perplejidad ≈ {torch.exp(torch.tensor(loss)):.1f}")
        ok = loss < 6.0 and len(text) > len(prompt)
        RESULTS.append({"Prueba": "GPT-2 124M · carga de pesos + generación",
                        "Resultado": "✅ OK" if ok else "❌ FALLO",
                        "Tiempo (s)": round(time.time() - t0, 1),
                        "Detalle": f"loss={loss:.2f} en {device}"})
        del model, params
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        import traceback; traceback.print_exc()
        RESULTS.append({"Prueba": "GPT-2 124M · carga de pesos + generación", "Resultado": "❌ FALLO",
                        "Tiempo (s)": round(time.time() - t0, 1), "Detalle": repr(e)[:120]})
else:
    print("Bloque desactivado (RUN_GPT2_DEMO = False)")

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 23.9kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 586kiB/s]
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 372kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [04:24<00:00, 1.88MiB/s]
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 21.0MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:01<00:00, 397kiB/s]
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 327kiB/s]



📝 Texto generado:
 Every effort moves you as far as the hand can go until the end of your turn. This means there's a very small delay before you lose to the card. This

Pérdida (cross-entropy) en texto de control: 3.80  ·  perplejidad ≈ 44.8


## 7. Bloque extendido (opcional)

Tests del paquete `llms_from_scratch` (todos los capítulos y apéndices) y de las implementaciones
desde cero de modelos modernos (Llama 3.2, Qwen3, Gemma 3, OLMo 3), que se comparan contra
`transformers` de Hugging Face. Es el bloque que más tarda.

Notas:
- Los tests de Qwen3 descargan el tokenizador desde Hugging Face, y el de Llama 3 crea un modelo de 1B parámetros
  (necesita varios GB de RAM; la RAM de Colab gratuito, ~12 GB, debería bastar). Cada fichero corre en su propio proceso.
- Un fallo aquí puede deberse a la versión de `transformers` que trae Colab y no al repositorio: revisa la salida.

In [11]:
if RUN_EXTENDED:
    # Mismos requisitos extra que usa el workflow de Linux del repo
    !pip install -q -r ch05/07_gpt_to_llama/tests/test-requirements-extra.txt
    # Un proceso por fichero: así se libera la RAM entre tests (el de Llama 3.2 1B ocupa varios GB)
    import glob
    for path in sorted(glob.glob("pkg/llms_from_scratch/tests/test_*.py")):
        mod = os.path.basename(path)[5:-3]
        run_step(f"Paquete · {mod}", f"python -m pytest -q {path}")
    extended = {
        "Bonus · RoPE y piezas de Llama": "ch05/07_gpt_to_llama/tests/tests_rope_and_parts.py",
        "Bonus · Llama 3.2 desde cero":   "ch05/07_gpt_to_llama/tests/test_llama32_nb.py",
        "Bonus · Qwen3 desde cero":       "ch05/11_qwen3/tests/test_qwen3_nb.py",
        "Bonus · Gemma 3 desde cero":     "ch05/12_gemma3/tests/test_gemma3_nb.py",
        "Bonus · Gemma 3 + KV-cache":     "ch05/12_gemma3/tests/test_gemma3_kv_nb.py",
        "Bonus · OLMo 3 desde cero":      "ch05/13_olmo3/tests/test_olmo3_nb.py",
        "Bonus · OLMo 3 + KV-cache":      "ch05/13_olmo3/tests/test_olmo3_kvcache_nb.py",
    }
    for name, path in extended.items():
        run_step(name, f"python -m pytest -q {path}")
else:
    print("Bloque desactivado (RUN_EXTENDED = False)")


▶ Paquete · appendix_a
$ python -m pytest -q pkg/llms_from_scratch/tests/test_appendix_a.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
.                                                                        [100%]
1 passed in 3.25s

▶ Paquete · appendix_d
$ python -m pytest -q pkg/llms_from_scratch/tests/test_appendix_d.py
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
.                                                                        [100%]
1 passed in 11.88s

▶ Pa

## 8. Resumen

In [12]:
import pandas as pd
from IPython.display import display, Markdown

df = pd.DataFrame(RESULTS)
n_ok   = df["Resultado"].str.startswith("✅").sum()
n_warn = df["Resultado"].str.startswith("⚠️").sum()
n_fail = df["Resultado"].str.startswith("❌").sum()

commit = subprocess.run("git log -1 --format=%h", shell=True, cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
veredicto = ("🟢 **El repositorio funciona en este entorno.**" if n_fail == 0 else
             "🔴 **Hay pruebas que fallan** — revisa la salida de las filas marcadas con ❌.")

display(Markdown(f"### {veredicto}\n\n"
                 f"Commit `{commit}` · PyTorch {torch.__version__} · {device_name}  \n"
                 f"✅ {n_ok} correctas · ⚠️ {n_warn} con avisos · ❌ {n_fail} fallos"))
display(df.style.hide(axis="index").set_properties(**{"text-align": "left"}))

### 🟢 **El repositorio funciona en este entorno.**

Commit `b5a5165` · PyTorch 2.11.0+cu130 · Tesla T4  
✅ 33 correctas · ⚠️ 0 con avisos · ❌ 0 fallos

Prueba,Resultado,Tiempo (s),Detalle
Check de entorno (requirements.txt),✅ OK,15.000000,jupyterlab ignorado (no aplica en Colab)
Cap. 4 · Modelo GPT,✅ OK,20.100000,1 passed in 12.84s
Cap. 5 · Preentrenamiento,✅ OK,46.700000,6 passed in 38.89s
Cap. 6 · Fine-tuning (spam),✅ OK,19.600000,1 passed in 15.84s
Cap. 2 · Dataloader (notebook),✅ OK,7.800000,"3 passed, 1 skipped in 3.57s"
Cap. 3 · Multi-head attention (notebook),✅ OK,8.300000,8 passed in 4.59s
Cap. 2 · Intuición del dataloader (nb),✅ OK,6.800000,11 passed in 3.29s
Cap. 3 · MHA eficiente (bonus),✅ OK,9.600000,4 passed in 4.25s
Cap. 4 · KV-cache (bonus),✅ OK,22.400000,9 passed in 17.50s
Cap. 2 · BPE desde cero (bonus),✅ OK,15.000000,12 passed in 11.44s
